<a href="https://colab.research.google.com/github/natalialungu-lab/Project_Cost_Tool/blob/main/project_cost_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M2U2 | Project Cost Analysis Tool
**Course:** M2U2 — Individual Assignment  
**Date:** 2026-06-21  
**Purpose:** Reads construction project data from a CSV file, calculates total cost per site (materials + labour), compares actual vs. estimated values, flags cost overruns and schedule delays, and exports a structured summary report.

---
### How to run in Google Colab:
1. Run **Cell 1** — uploads your CSV file from your computer
2. Run **Cells 2–7** — loads all functions
3. Run **Cell 8** — executes the full analysis and saves reports

## Step 1 — Upload CSV File (Google Colab)

In [1]:
# Upload the project_data.csv file from your local computer
# A file picker dialog will appear after running this cell
from google.colab import files
uploaded = files.upload()
print("File uploaded successfully:", list(uploaded.keys()))

Saving project_data.csv to project_data (3).csv
File uploaded successfully: ['project_data (3).csv']


## Step 2 — Import Libraries & Configure File Paths

In [2]:
# Standard Python library for reading and writing CSV files
import csv
from pathlib import Path

# ── FILE PATHS ─────────────────────────────────────────────────────────────────
# Input CSV supplied by the lecturer containing site-level cost and duration data
INPUT_CSV  = Path("project_data.csv")

# Output files: summary table (CSV) and formatted report (TXT)
OUTPUT_CSV = Path("project_cost_summary.csv")
OUTPUT_TXT = Path("project_summary_report.txt")

print(f"[INFO] Input file : {INPUT_CSV}")
print(f"[INFO] Output CSV : {OUTPUT_CSV}")
print(f"[INFO] Output TXT : {OUTPUT_TXT}")

[INFO] Input file : project_data.csv
[INFO] Output CSV : project_cost_summary.csv
[INFO] Output TXT : project_summary_report.txt


## Step 3 — CSV Handling Function

In [3]:
def load_project_data(filepath: Path) -> list:
    """
    Loads structured construction project data from a CSV file.
    Each row represents one construction site.
    Raises FileNotFoundError if the file does not exist.
    Raises ValueError if the file is empty or has no header row.
    """
    # Verify the file exists before attempting to open it
    if not filepath.exists():
        raise FileNotFoundError(
            f"Project data file not found: {filepath}\n"
            f"Ensure the CSV file is uploaded in Step 1."
        )

    rows = []
    with filepath.open("r", newline="", encoding="utf-8-sig") as csvfile:
        reader = csv.DictReader(csvfile)

        # Confirm the CSV contains a valid header row
        if reader.fieldnames is None:
            raise ValueError("CSV file is empty or missing a header row.")

        # Read each project row into a dictionary
        for row in reader:
            rows.append(row)

    # Confirm that data rows exist below the header
    if not rows:
        raise ValueError("CSV file contains a header row but no project data.")

    return rows

print("[OK] load_project_data() defined.")

[OK] load_project_data() defined.


## Step 4 — Calculation Functions (Python Logic + Modular Design)

In [4]:
def calculate_total_cost(material_cost: float, labour_cost: float) -> float:
    """
    Calculates total direct construction cost for a site.
    Formula: Total Cost = Material Cost + Labour Cost
    """
    return material_cost + labour_cost


def calculate_cost_variance(estimated: float, actual: float) -> float:
    """
    Calculates cost variance using the EVM (Earned Value Management) formula.
    Formula: Cost Variance = Actual Cost - Estimated Cost
    CV > 0 => Cost overrun | CV < 0 => Cost saving | CV = 0 => On budget
    """
    return actual - estimated


def calculate_schedule_variance(planned_days: float, actual_days: float) -> float:
    """
    Calculates schedule variance in construction programme management.
    Formula: Schedule Variance = Actual Duration - Planned Duration
    SV > 0 => Delayed | SV < 0 => Ahead of programme | SV = 0 => On programme
    """
    return actual_days - planned_days


def forecast_cost_at_completion(actual_total: float,
                                actual_duration: float,
                                schedule_variance: float) -> float:
    """
    Forecasts Estimate at Completion (EAC) if delay trend continues.
    Based on EVM principles (BS EN ISO 21508).
    Formula: EAC = Actual Cost + (Daily Burn Rate x Schedule Variance)
    """
    # No delay or invalid duration: return actual cost unchanged
    if actual_duration <= 0 or schedule_variance <= 0:
        return actual_total

    # Derive the daily cost burn rate from current expenditure
    daily_burn_rate = actual_total / actual_duration

    # Project additional cost based on outstanding delay
    return actual_total + (daily_burn_rate * schedule_variance)


def assess_project_status(cost_variance: float, schedule_variance: float) -> str:
    """
    Assesses site performance — mirrors RAG status reporting in
    structural engineering project management.
    Returns: 'COST OVERRUN', 'POTENTIAL DELAY', both, or 'ON TRACK'.
    """
    flags = []

    # Flag if actual cost exceeds the original estimate
    if cost_variance > 0:
        flags.append("COST OVERRUN")

    # Flag if actual duration exceeds the planned programme
    if schedule_variance > 0:
        flags.append("POTENTIAL DELAY")

    return "; ".join(flags) if flags else "ON TRACK"


def analyse_single_project(row: dict) -> dict:
    """
    Processes one construction site record.
    Extracts cost and duration data, calls calculation functions,
    and returns a consolidated KPI dictionary for reporting.
    """
    try:
        # Extract estimated (budgeted) cost components
        est_material = float(row["Estimated_Material_Cost"])
        est_labour   = float(row["Estimated_Labor_Cost"])

        # Extract actual (incurred) cost components
        act_material = float(row["Actual_Material_Cost"])
        act_labour   = float(row["Actual_Labor_Cost"])

        # Extract programme durations in calendar days
        actual_days  = float(row["Actual_Duration_Days"])
        planned_days = float(row["Planned_Duration_Days"])

    except KeyError as e:
        raise KeyError(f"Required column missing from CSV: {e}") from e
    except ValueError as e:
        raise ValueError(f"Non-numeric value in row {row.get('Project', '?')}: {e}") from e

    # Calculate total estimated and actual construction costs
    estimated_total = calculate_total_cost(est_material, est_labour)
    actual_total    = calculate_total_cost(act_material, act_labour)

    # Calculate cost and schedule variances
    cost_var     = calculate_cost_variance(estimated_total, actual_total)
    schedule_var = calculate_schedule_variance(planned_days, actual_days)

    # Assess status and forecast cost at completion
    status   = assess_project_status(cost_var, schedule_var)
    forecast = forecast_cost_at_completion(actual_total, actual_days, schedule_var)

    return {
        "Project"                    : row["Project"],
        "Site"                       : row["Site"],
        "Estimated_Total_Cost"       : round(estimated_total, 2),
        "Actual_Total_Cost"          : round(actual_total, 2),
        "Cost_Variance_EUR"          : round(cost_var, 2),
        "Planned_Duration_Days"      : round(planned_days, 2),
        "Actual_Duration_Days"       : round(actual_days, 2),
        "Schedule_Variance_Days"     : round(schedule_var, 2),
        "Forecast_Cost_At_Completion": round(forecast, 2),
        "Status"                     : status,
    }


def analyse_all_projects(rows: list) -> list:
    """Iterates over all project rows and returns a list of analysed results."""
    results = []
    # Loop through every construction site record
    for row in rows:
        result = analyse_single_project(row)
        results.append(result)
    return results

print("[OK] All calculation functions defined.")

[OK] All calculation functions defined.


## Step 5 — Export Functions (Analysis & Output)

In [5]:
def export_to_csv(results: list, filepath: Path) -> None:
    """
    Exports the full project analysis results to a CSV file.
    Formatted for use in construction cost reports or QS software.
    """
    with filepath.open("w", newline="", encoding="utf-8") as f:
        # Write header row using dictionary keys as column names
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        # Write one data row per construction site
        writer.writerows(results)
    print(f"[OK] CSV summary report exported  =>  {filepath}")


def export_to_txt(results: list, filepath: Path) -> None:
    """
    Exports a structured plain-text cost report to a .txt file.
    Format is consistent with standard construction project progress reports.
    """
    # Portfolio-level aggregates
    total_estimated    = sum(r["Estimated_Total_Cost"] for r in results)
    total_actual       = sum(r["Actual_Total_Cost"] for r in results)
    total_overruns     = sum(1 for r in results if r["Cost_Variance_EUR"] > 0)
    total_delays       = sum(1 for r in results if r["Schedule_Variance_Days"] > 0)
    portfolio_variance = total_actual - total_estimated

    lines = [
        "=" * 56,
        "   CONSTRUCTION PROJECT COST SUMMARY REPORT",
        "   Generated by: M2U2 Project Cost Analysis Tool",
        "=" * 56,
        f"  Total sites reviewed          : {len(results)}",
        f"  Sites with cost overrun       : {total_overruns}",
        f"  Sites with schedule delay     : {total_delays}",
        f"  Portfolio estimated cost      : EUR {total_estimated:>12,.2f}",
        f"  Portfolio actual cost         : EUR {total_actual:>12,.2f}",
        f"  Portfolio cost variance       : EUR {portfolio_variance:>12,.2f}",
        "=" * 56,
        "",
        "  SITE-BY-SITE BREAKDOWN",
        "  " + "-" * 54,
    ]

    for r in results:
        # Highlight sites with overruns or delays
        status_line = (
            f"  !!! {r['Status']} !!!"
            if r["Status"] != "ON TRACK"
            else "  Status : ON TRACK — within budget and programme"
        )
        lines += [
            f"  Project : {r['Project']}   |   Site : {r['Site']}",
            f"  Estimated total cost            : EUR {r['Estimated_Total_Cost']:>10,.2f}",
            f"  Actual total cost               : EUR {r['Actual_Total_Cost']:>10,.2f}",
            f"  Cost variance (overrun/saving)  : EUR {r['Cost_Variance_EUR']:>10,.2f}",
            f"  Planned construction duration   : {r['Planned_Duration_Days']:>6} days",
            f"  Actual construction duration    : {r['Actual_Duration_Days']:>6} days",
            f"  Schedule variance               : {r['Schedule_Variance_Days']:>6} days",
            f"  Forecast cost at completion     : EUR {r['Forecast_Cost_At_Completion']:>10,.2f}",
            status_line,
            "  " + "-" * 54,
        ]

    filepath.write_text("\n".join(lines), encoding="utf-8")
    print(f"[OK] TXT summary report exported   =>  {filepath}")

print("[OK] Export functions defined.")

[OK] Export functions defined.


## Step 6 — Run Full Analysis (with Error Handling)

In [6]:
# ── MAIN EXECUTION — runs the full analysis pipeline ──────────────────────────
try:
    print(f"[INFO] Loading construction project data from: {INPUT_CSV}")
    rows = load_project_data(INPUT_CSV)
    print(f"[INFO] Successfully loaded {len(rows)} project site records.\n")

    # Run cost and schedule analysis for all sites
    results = analyse_all_projects(rows)

    # Print summary table to terminal
    print("=" * 80)
    print("   CONSTRUCTION PROJECT COST PERFORMANCE SUMMARY")
    print("=" * 80)
    print(f"{'Project':<10} {'Site':<8} {'Est. Cost':>12} {'Act. Cost':>12} {'Cost Var':>11} {'Sched Var':>11}  Status")
    print("-" * 80)
    for r in results:
        print(
            f"{r['Project']:<10} {r['Site']:<8} "
            f"{r['Estimated_Total_Cost']:>12,.2f} "
            f"{r['Actual_Total_Cost']:>12,.2f} "
            f"{r['Cost_Variance_EUR']:>11,.2f} "
            f"{r['Schedule_Variance_Days']:>8.0f} days   "
            f"{r['Status']}"
        )
    print("=" * 80)

    # Export reports to CSV and TXT
    export_to_csv(results, OUTPUT_CSV)
    export_to_txt(results, OUTPUT_TXT)

    print("\n[COMPLETE] All project cost reports generated successfully.")

except FileNotFoundError as e:
    print(f"[ERROR] Input file not found    : {e}")
except KeyError as e:
    print(f"[ERROR] Missing CSV column       : {e}")
except ValueError as e:
    print(f"[ERROR] Invalid data in CSV      : {e}")
except Exception as e:
    print(f"[ERROR] Unexpected error         : {e}")

[INFO] Loading construction project data from: project_data.csv
[INFO] Successfully loaded 5 project site records.

   CONSTRUCTION PROJECT COST PERFORMANCE SUMMARY
Project    Site        Est. Cost    Act. Cost    Cost Var   Sched Var  Status
--------------------------------------------------------------------------------
P001       Site A      22,100.00    23,000.00      900.00        5 days   COST OVERRUN; POTENTIAL DELAY
P002       Site B      18,000.00    18,000.00        0.00        0 days   ON TRACK
P003       Site C      25,500.00    27,000.00    1,500.00        5 days   COST OVERRUN; POTENTIAL DELAY
P004       Site D      14,700.00    14,000.00     -700.00       -5 days   ON TRACK
P005       Site E      29,000.00    31,000.00    2,000.00       10 days   COST OVERRUN; POTENTIAL DELAY
[OK] CSV summary report exported  =>  project_cost_summary.csv
[OK] TXT summary report exported   =>  project_summary_report.txt

[COMPLETE] All project cost reports generated successfully.


## Step 7 — Download Output Files

In [7]:
# Download the generated summary reports to your local computer
from google.colab import files
files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_TXT))
print("[OK] Output files downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] Output files downloaded.
